# Hidden Value: Points Projection, ADVANCED (trial notebook)

**Purpose:** test additions to the points projection and decide whether they are worth keeping. The base notebook (WEEKLY_2) is not changed. This one saves to its own files, so the versions can be graded side by side.

**The six additions**
1. **Quarterback history.** The expected starter's EPA per dropback over the previous three seasons, pulled toward league average.
2. **Non-QB injuries.** Regular starters (50%+ of snaps) who are Out, Doubtful or on a reserve list, counted by how much playing time the team's ratings include for them.
3. **Market starting point.** The average points the betting line implied for and against each team in its earlier games this season. This borrows the market's opinion, so it is kept as a separate version.
4. **Wind.** Miles per hour above 10 in outdoor games.
5. **Last season.** How each offense and each defense played last season, with a weight that fades as this season's games pile up.
6. **Pace.** Drives per game for the offense, and drives per game faced by the opposing defense.
7. **Rain.** Rain forecast at kickoff in an outdoor game. Worth about -3.3 points for each team.

**Four projections are produced for every game**
- **Base:** the original model, unchanged.
- **Advanced:** base plus additions 1, 2 and 4.
- **Advanced 2:** Advanced plus additions 5, 6 and 7. Like Advanced, everything in it comes from football data, nothing from betting lines.
- **Advanced + market:** Advanced plus additions 3 and 7. This is the most accurate version and the headline number.

**What the backtest says (2021-2025, weeks 4-18, 2,238 team-games, each season predicted from the other four)**

| Version | Avg miss per team (pts) | Better than base by | Seasons better (of 5) |
|---|---|---|---|
| Base | 7.50 | | |
| Base + QB history only | 7.49 | 0.02 | 4 |
| Base + injuries only | 7.46 | 0.04 | 4 |
| Base + market start only | 7.46 | 0.04 | 4 |
| Base + wind only | 7.48 | 0.03 | 4 |
| Base + last season only | 7.47 | 0.03 | 5 |
| Base + pace only | 7.50 | 0.01 | 2 |
| **Advanced (1 + 2 + 4)** | **7.42** | **0.08** | **5** |
| **Advanced 2 (1 + 2 + 4 + 5 + 6)** | **7.40** | **0.10** | **5** |
| **Advanced + market (1 to 4)** | **7.38** | **0.12** | **5** |
| Advanced 2 + rain | 7.37 | 0.13 | 5 |
| **Advanced + market + rain** | **7.36** | **0.15** | **5** |
| Betting market | 7.25 | 0.25 | |

Read that honestly. No single addition is a clear win by itself: each one is small enough that it could be noise. Together they are a real improvement, better in all five seasons. Advanced 2 closes about 40% of the gap between Base and the market. No version beat the spread in the backtest (about 50%), so this is a better forecast, not a betting edge.

**Second check, on seasons the model had never been run on.** The weights were frozen from 2021-2025 and used to project 2016-2020 (2,082 team-games). Average miss: Base 7.54, Advanced 7.50, Advanced 2 7.41, betting market 7.27. Last season helped in all five of those seasons, and most of its gain comes in weeks 4 to 6. That check was run outside this notebook and is recorded in the project note `backtest-results-2026-10-10`.

**Rain (added Oct 19).** Rain was the only one of about 40 extra ideas that passed every check: better in 9 of 10 seasons across 2016-2025, in both halves of the decade. It moves totals, not winners, because it lowers both teams' scores. The backtest uses the weather recorded at the game; live projections use a forecast, so expect a smaller gain.

**Tested and not adopted** (no gain, or a gain in one period only): EPA per play, success rate, turnover adjustments, starting field position, injuries split by position, counting Questionable players, weighting recent games more, and fitting the weights on ten seasons instead of five.

**How to decide whether to keep it:** Section 10 saves all four projections before kickoff and Section 11 grades them on the same games. Keep a version if it stays ahead of the one before it over about 100 graded team scores.

**Change log**
- 2026-11-09: saving is now per game (`SAVE_NOW` in Section 10), so Thursday's game can be saved Thursday and the rest after Sunday inactives. Added candidate additions under test in Section 12: EPA per play, success rate, recent form, red-zone luck, turnover luck, and a blend with this week's line. Several of these were tested before and not adopted (see above); this re-test runs them on top of Advanced + market + rain. Live projections do not change until `COEF_ADV3` or `BLEND_W` is filled in from the backtest. Grading now also shows each version's average bias (projected minus actual).
- 2026-10-19: added rain (addition 7) to Advanced 2 and Advanced + market and refitted both on 2021-2025 (no 2026 result used). Advanced 2 + rain 7.367, Advanced + market + rain 7.356, against 7.399 and 7.384 without it. Weeks 5 and 6 were saved before this change. Base and Advanced are unchanged. Grading now leaves out tied games and adds a record of games where Advanced + market and the line pick different winners.
- 2026-10-10: added Advanced 2 (last season and pace). Its weights were fitted on 2021-2025 only. No 2026 result was used. Base, Advanced and Advanced + market are unchanged. The injury-report warning in Section 7 was corrected: it used to fire when half the listed players had no game status, which is normal on a final report.

## 1. Setup and load

Same loads as the base notebook, plus three seasons of player stats (quarterback history), this season's snap counts, injury reports and weekly rosters (missing starters), and last season's play-by-play (last-season starting point).

In [1]:
import nflreadpy as nfl
import pandas as pd, numpy as np
import os, glob

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 240)

SEASON = 2026
FOLDER = r"C:\Users\brade\OneDrive - Worcester Polytechnic Institute (wpi.edu)\Desktop\Football"
CSV = os.path.join(FOLDER, "hidden_value_2026_drives.csv")

def to_pd(x, cols=None):
    """nflreadpy returns Polars; convert to pandas. cols keeps memory down on big pulls."""
    if cols is not None and hasattr(x, "select"):
        x = x.select([c for c in cols if c in x.columns])
    try:
        return x.to_pandas()
    except AttributeError:
        return x if cols is None else x[[c for c in cols if c in x.columns]]

drives = pd.read_csv(CSV)

PBP_COLS = ["game_id", "season", "season_type", "week", "posteam", "defteam", "fixed_drive",
            "fixed_drive_result", "play_type", "qb_kneel", "qb_spike", "qb_dropback", "id", "name", "play_id",
            "epa", "success", "interception", "fumble_lost", "yardline_100"]   # last five: candidate additions
STAT_COLS = ["player_id", "player_display_name", "season", "week", "season_type", "attempts", "sacks_suffered", "passing_epa"]
ROSTER_COLS = ["season", "team", "week", "game_type", "full_name", "status"]
SCHED_COLS = ["game_id", "season", "game_type", "week", "away_team", "home_team", "away_score", "home_score", "location",
              "spread_line", "total_line", "roof", "wind"]

def load_reg(loader, seasons, cols=None):
    """One nflreadpy pull, regular season only."""
    x = to_pd(loader(seasons=seasons), cols)
    key = "game_type" if "game_type" in x.columns else "season_type"
    return x[x[key] == "REG"].copy()

pbp = load_reg(nfl.load_pbp, [SEASON], PBP_COLS)
sched = load_reg(nfl.load_schedules, [SEASON], SCHED_COLS)

# Games the schedule file lists as a home game but that are played at a neutral site.
# Listed by game_id, so entries can stay here all season.
NEUTRAL_SITE_GAMES = ["2026_05_PHI_JAX"]   # Eagles vs Jaguars, London
sched.loc[sched["game_id"].isin(NEUTRAL_SITE_GAMES), "location"] = "Neutral"
hist_stats = load_reg(nfl.load_player_stats, [SEASON - 3, SEASON - 2, SEASON - 1], STAT_COLS)
snaps = load_reg(nfl.load_snap_counts, [SEASON])
inj = load_reg(nfl.load_injuries, [SEASON])
rosters = load_reg(nfl.load_rosters_weekly, [SEASON], ROSTER_COLS)
prev_pbp = load_reg(nfl.load_pbp, [SEASON - 1], PBP_COLS)      # last season, for the last-season starting point

# Week being projected = the next week with unplayed games. Set WEEK by hand to project a different one.
WEEK = int(sched.loc[sched["home_score"].isna(), "week"].min())
print("projecting week", WEEK, "using drives from weeks before it")

csv_games, pbp_games = set(drives["game_id"]), set(pbp["game_id"])
missing = sorted(pbp_games - csv_games)
if missing:
    print(">> STALE CSV: re-run Phase 2 first. Games missing from the drive table:", missing)
else:
    print("drive table is current through week", int(drives["week"].max()), "|", len(drives), "drives")
print("history seasons for QBs:", sorted(hist_stats["season"].unique().tolist()),
      "| snap weeks:", sorted(snaps["week"].unique().tolist()), "| injury report weeks:", sorted(inj["week"].unique().tolist()))
print("last season loaded:", SEASON - 1, "|", prev_pbp["game_id"].nunique(), "games")

projecting week 9 using drives from weeks before it
>> STALE CSV: re-run Phase 2 first. Games missing from the drive table: ['2026_09_ARI_SEA', '2026_09_CIN_ATL', '2026_09_CLE_NO', '2026_09_DAL_IND', '2026_09_DEN_CAR', '2026_09_DET_MIA', '2026_09_GB_NE', '2026_09_HOU_LAC', '2026_09_JAX_BAL', '2026_09_LA_WAS', '2026_09_LV_SF', '2026_09_NYG_PHI', '2026_09_NYJ_KC', '2026_09_TB_CHI']
history seasons for QBs: [2023, 2024, 2025] | snap weeks: [1, 2, 3, 4, 5, 6, 7, 8, 9] | injury report weeks: [1, 2, 3, 4, 5, 6, 7, 8, 9]
last season loaded: 2025 | 272 games


## 2. Constants (set on 2021-2025, do not tune on 2026)

`COEF` is the original model. `COEF_ADV`, `COEF_ADV2` and `COEF_ADV_MKT` are the trial versions. Section 12 re-derives all four.

In [2]:
W_OTHER_QB = 0.25   # weight on drives run by a QB other than the expected starter
K_OFF = 20          # league-average drives added to every offense rating
K_DEF = 20          # league-average drives added to every defense rating
K_QB = 250          # league-average dropbacks added to a quarterback's history
STARTER_SHARE = 0.50   # a regular starter averages at least this share of snaps when he plays
WIND_FLOOR = 10        # wind only counts above this many mph
K_PACE = 3             # league-average games added to every team's drives per game
FADE_K, FADE_PER_WEEK = 20, 10.3   # last season's weight = 20 / (10.3 x weeks played + 20): 0.33 in week 5, 0.11 in week 17

# Candidate additions under test (Section 12). Live projections ignore them until COEF_ADV3 / BLEND_W are filled in.
K_PLAYS = 200    # league-average plays added to the EPA and success-rate ratings
K_RZ = 15        # league-average red-zone trips added to the red-zone rating
K_TO = 40        # league-average drives added to the turnover rating
DECAY = 0.85     # recent form: a drive from n weeks before the last game counts DECAY**n (0.85 -> 4 weeks back counts 0.52)
COEF_ADV3 = None # paste from Section 12 ONLY if Advanced 3 beats Advanced + market + rain there
BLEND_W = None   # paste from Section 12 ONLY if the blend beats the line alone there, e.g. {"const": .., "model": .., "market": ..}

# points = const + sum(weight x input). Inputs are described where they are built.
COEF = {"const": 4.41, "off": 6.22, "deff": 2.47, "home": 1.15, "new_qb": -3.38}
COEF_ADV = {"const": 6.30, "off": 5.40, "deff": 2.26, "home": 1.12, "new_qb": -2.70,
            "qb_prior": 7.48, "qb_nohist": -1.21, "off_lost": -0.64, "def_lost": 1.29, "wind_over": -0.35}
COEF_ADV2 = {"const": -9.971, "off": 5.778, "deff": 2.272, "home": 1.124, "new_qb": -2.833,
             "qb_prior": 5.872, "qb_nohist": -1.073, "off_lost": -0.534, "def_lost": 1.270, "wind_over": -0.318,
             "prev_o_w": 3.798, "prev_d_w": 10.507, "pace": 0.769, "rain": -3.347}   # three decimals: pace is a number near 20, so rounding matters
COEF_ADV_MKT = {"const": -3.614, "off": 4.658, "deff": 0.287, "home": 1.139, "new_qb": -2.567,
                "qb_prior": 5.532, "qb_nohist": -1.084, "off_lost": -0.652, "def_lost": 1.353, "wind_over": -0.323,
                "mkt_for": 0.192, "opp_mkt_against": 0.513, "rain": -3.374}
# Before Oct 19 (no rain), for reference:
# COEF_ADV2    = {"const": -9.940, "off": 5.749, "deff": 2.355, "home": 1.123, "new_qb": -2.870, "qb_prior": 5.942, "qb_nohist": -1.067,
#                 "off_lost": -0.580, "def_lost": 1.262, "wind_over": -0.340, "prev_o_w": 3.693, "prev_d_w": 10.355, "pace": 0.758}
# COEF_ADV_MKT = {"const": -3.62, "off": 4.62, "deff": 0.43, "home": 1.14, "new_qb": -2.61, "qb_prior": 5.52, "qb_nohist": -1.07,
#                 "off_lost": -0.70, "def_lost": 1.34, "wind_over": -0.35, "mkt_for": 0.19, "opp_mkt_against": 0.50}

def points(feat, coef):
    """feat: DataFrame of inputs, one row per team. Returns projected points."""
    return coef["const"] + sum(w * feat[k] for k, w in coef.items() if k != "const")

## 3. Who ran each drive

One QB per drive: whoever took the most dropbacks on it. A drive with no dropbacks takes the QB from the team's nearest drive in that game.

In [3]:
def drive_qb(pbp):
    p = pbp[pbp["posteam"].notna() & pbp["fixed_drive"].notna()].copy()
    p["drive_id"] = p["game_id"].astype(str) + "_" + p["fixed_drive"].astype(int).astype(str)
    db = p[(p["qb_dropback"] == 1) & p["id"].notna()]
    top = (db.groupby(["drive_id", "id", "name"]).size().rename("n").reset_index()
             .sort_values(["n", "id"]).drop_duplicates("drive_id", keep="last"))
    return top.set_index("drive_id")[["id", "name"]].rename(columns={"id": "qb_id", "name": "qb"})

d = drives.drop(columns=[c for c in ["qb_id", "qb"] if c in drives.columns]).join(drive_qb(pbp), on="drive_id")
d = d.sort_values(["game_id", "posteam", "fixed_drive"])
for c in ["qb_id", "qb"]:
    d[c] = d.groupby(["game_id", "posteam"])[c].ffill()
    d[c] = d.groupby(["game_id", "posteam"])[c].bfill()
print("drives with no QB assigned (should be 0):", int(d["qb_id"].isna().sum()))

# forecasting frame: every drive from weeks BEFORE the one being projected, except clock-expiration
# drives (garbage time INCLUDED, see intro). The week filter stops a game from helping to predict itself.
f = d[(d["clock_expiration"] == 0) & (d["week"] < WEEK)].copy()
print("drives in forecasting frame:", len(f), "| fewest on one team:", int(f.groupby("posteam").size().min()))

drives with no QB assigned (should be 0): 0
drives in forecasting frame: 2453 | fewest on one team: 63


## 4. Expected starters

Default = the QB who ran the most drives in each team's most recent game. **Check this table every week** and put changes in `STARTER_OVERRIDES` (team -> QB name exactly as it appears in the `qb` column, e.g. `"WAS": "J.Daniels"`). A QB who has not played for the team yet is allowed; he just gets a share of 0.

The schedule file has QB columns too, but they disagree with the play-by-play in several 2026 games, so they are not used here.

In [4]:
STARTER_OVERRIDES = {   
     "BAL": "J.Fagnano",   # Lamar and Huntley on IR
    "CHI": "C.Williams",
    "ATL": "M.Penix",
    "PHI": "J.Milton",    # McKee benched
    "TB": "B.Mayfield",
    "NYG": "J.McCarthy",
    "CLE": "S.Sanders",   # Watson in concussion protocol
    "SEA": "D.Lock",      # Darnold out, quad
    "MIN": "C.Wentz",     # Murray out, hamstring
    "ARI": "G.Minshew",   # Brissett benched, then traded
}

def expected_starters(f, overrides=None):
    last_wk = f.groupby("posteam")["week"].transform("max")
    lg = f[f["week"] == last_wk]
    s = (lg.groupby(["posteam", "qb"]).size().rename("n").reset_index()
           .sort_values(["n", "qb"]).drop_duplicates("posteam", keep="last").set_index("posteam")["qb"])
    for team, qb in (overrides or {}).items():
        if team not in s.index:
            raise ValueError(f"override team {team!r} is not a team in the drive table")
        if qb not in set(f.loc[f["posteam"] == team, "qb"]):
            print(f"   note: {qb} has no drives for {team} this season, so he is treated as a brand-new starter")
        s[team] = qb
    return s

starters = expected_starters(f, STARTER_OVERRIDES)

own = f[f["qb"] == f["posteam"].map(starters)]
tbl = pd.DataFrame({"expected_starter": starters,
                    "his_drives": own.groupby("posteam").size().reindex(starters.index).fillna(0).astype(int),
                    "team_drives": f.groupby("posteam").size(),
                    "last_week_he_played": own.groupby("posteam")["week"].max().reindex(starters.index)})
tbl["qb_share"] = (tbl["his_drives"] / tbl["team_drives"]).round(2)
print("teams where the expected starter has NOT run every drive (these are the ones to double-check):")
print(tbl[tbl["qb_share"] < 0.999].sort_values("qb_share").to_string())

   note: J.Milton has no drives for PHI this season, so he is treated as a brand-new starter
   note: G.Minshew has no drives for ARI this season, so he is treated as a brand-new starter
teams where the expected starter has NOT run every drive (these are the ones to double-check):
        expected_starter  his_drives  team_drives  last_week_he_played  qb_share
posteam                                                                         
ARI            G.Minshew           0           87                  NaN      0.00
PHI             J.Milton           0           81                  NaN      0.00
CLE            S.Sanders           1           84                  8.0      0.01
BAL            J.Fagnano           8           85                  8.0      0.09
SEA               D.Lock          17           88                  2.0      0.19
NYG           J.McCarthy          15           74                  7.0      0.20
MIN              C.Wentz          18           70                  2.0

## 5. Ratings

In [5]:
def ratings(f, starters, w_other=W_OTHER_QB, k_off=K_OFF, k_def=K_DEF):
    lg = f["offensive_points"].mean()
    mine = f["qb"] == f["posteam"].map(starters)
    wt = np.where(mine, 1.0, w_other)
    g = f.assign(wt=wt, wpts=wt * f["offensive_points"], mine=mine.astype(int)).groupby("posteam")
    r = pd.DataFrame({"drives": g.size(), "off_ppd": g["offensive_points"].mean(),
                      "qb_share": g["mine"].mean(),
                      "off_rating": (g["wpts"].sum() + k_off * lg) / (g["wt"].sum() + k_off)})
    gd = f.groupby("defteam")
    r["drives_faced"] = gd.size()
    r["def_ppd"] = gd["offensive_points"].mean()
    r["def_rating"] = (gd["offensive_points"].sum() + k_def * lg) / (gd.size() + k_def)
    r["expected_starter"] = starters
    return r, lg

R, LEAGUE_PPD = ratings(f, starters)
print(f"league points per drive: {LEAGUE_PPD:.3f}")
print("off_ppd / def_ppd = raw.  off_rating / def_rating = what the projection uses (QB-weighted, pulled to average).")
print(R.sort_values("off_rating", ascending=False).round(2).to_string())

league points per drive: 2.234
off_ppd / def_ppd = raw.  off_rating / def_rating = what the projection uses (QB-weighted, pulled to average).
         drives  off_ppd  qb_share  off_rating  drives_faced  def_ppd  def_rating expected_starter
posteam                                                                                           
BUF          69     3.39      1.00        3.13            71     2.55        2.48          J.Allen
DAL          74     2.97      1.00        2.82            73     2.93        2.78       D.Prescott
SF           65     2.97      1.00        2.80            66     2.11        2.14          B.Purdy
CIN          67     2.96      1.00        2.79            67     2.24        2.24         J.Burrow
JAX          67     2.81      0.96        2.75            66     1.68        1.81       T.Lawrence
DET          71     2.87      1.00        2.73            69     2.65        2.56           J.Goff
CHI          81     2.64      0.64        2.72            81     1

## 6. Addition 1: quarterback history

For each quarterback: EPA per dropback over the previous three regular seasons, measured against the league rate in those seasons, with 250 league-average dropbacks added. So a veteran with 1,500 dropbacks keeps most of his number and a player with 80 is pulled almost to zero. A starter with under 100 prior dropbacks also gets the `qb_nohist` flag, because rookies and career backups score less than their (near-zero) rating suggests.

In [6]:
def qb_history(stats, k=K_QB):
    """stats: weekly player stats for the seasons that count as history. One row per quarterback."""
    s = stats[stats["attempts"] > 0].copy()
    s["db"] = s["attempts"] + s["sacks_suffered"]
    lg = s.groupby("season")[["passing_epa", "db"]].sum()
    s["expected"] = s["season"].map(lg["passing_epa"] / lg["db"]) * s["db"]
    a = s.groupby("player_id").agg(prior_db=("db", "sum"), epa=("passing_epa", "sum"), expected=("expected", "sum"))
    a["qb_prior"] = (a["epa"] - a["expected"]) / (a["prior_db"] + k)
    return a[["prior_db", "qb_prior"]]

def qb_inputs(qb_ids, hist):
    """qb_ids: Series team -> quarterback id. Returns qb_prior and qb_nohist per team."""
    out = pd.DataFrame(index=qb_ids.index)
    out["prior_db"] = qb_ids.map(hist["prior_db"]).fillna(0)
    out["qb_prior"] = qb_ids.map(hist["qb_prior"]).fillna(0.0)
    out["qb_nohist"] = (out["prior_db"] < 100).astype(float)
    return out

HIST = qb_history(hist_stats)

# expected starter's ID: from his drives for this team, else from any drive this season with that name
by_team = f.groupby(["posteam", "qb"])["qb_id"].first()
by_name = d.groupby("qb")["qb_id"].agg(lambda s: s.iloc[0] if s.nunique() == 1 else None)
def starter_id(team, qb):
    if (team, qb) in by_team.index:
        return by_team[(team, qb)]
    return by_name.get(qb, None)
starter_ids = pd.Series({t: starter_id(t, q) for t, q in starters.items()})
no_id = starter_ids[starter_ids.isna()].index.tolist()
if no_id:
    print("no player ID found for the starter on:", no_id, "(treated as having no history)")

QB = qb_inputs(starter_ids, HIST)
QB["starter"] = starters
QB["points_effect"] = COEF_ADV["qb_prior"] * QB["qb_prior"] + COEF_ADV["qb_nohist"] * QB["qb_nohist"]
print("quarterback history, best to worst. points_effect is what it adds in the Advanced version.")
print(QB.sort_values("qb_prior", ascending=False)[["starter", "prior_db", "qb_prior", "qb_nohist", "points_effect"]].round(3).to_string())

no player ID found for the starter on: ['ARI', 'PHI'] (treated as having no history)
quarterback history, best to worst. points_effect is what it adds in the Advanced version.
        starter  prior_db  qb_prior  qb_nohist  points_effect
SF      B.Purdy    1253.0     0.172        0.0          1.290
DET      J.Goff    1821.0     0.154        0.0          1.149
BUF     J.Allen    1600.0     0.141        0.0          1.058
GB       J.Love    1508.0     0.132        0.0          0.990
MIA    M.Willis     109.0     0.111        0.0          0.829
DAL  D.Prescott    1567.0     0.109        0.0          0.814
LA   M.Stafford    1716.0     0.102        0.0          0.761
NE       D.Maye     911.0     0.093        0.0          0.698
CIN    J.Burrow    1365.0     0.077        0.0          0.576
KC    P.Mahomes    1777.0     0.076        0.0          0.565
TB   B.Mayfield    1795.0     0.062        0.0          0.466
LAC   J.Herbert    1596.0     0.043        0.0          0.324
DEN       B.Nix   

## 7. Addition 2: regular starters who are unavailable

A regular starter averages 50%+ of his unit's snaps in the games he has played this season (quarterbacks are excluded, they are handled above). He counts as unavailable for the coming game if the injury report lists him Out or Doubtful, or his roster status is a reserve list.

What matters is playing time the ratings include that is now missing. A starter who played every game and is out this week counts in full. One who has been out all season counts for nothing, because the team's numbers already reflect life without him.

**Timing matters.** Final game statuses post on Friday afternoon (Saturday for Monday games). Run this section after that, or most injured players will not be counted yet.

About half the players on a finished injury report carry no game status, because they were cleared to play. So "no game status" on half the list is normal. The warning below only fires when almost nobody has a status, which means the report is not final.

In [7]:
def norm_name(s):
    return (s.astype(str).str.lower().str.replace(r"[^a-z ]", "", regex=True)
             .str.replace(r"\b(jr|sr|ii|iii|iv|v)\b", "", regex=True).str.split().str.join(" "))

def missing_starters(snaps, inj, rosters, week, min_share=STARTER_SHARE):
    """snaps / inj / rosters: one regular season. Returns (per-team totals, per-player detail)."""
    out = set()
    ij = inj[(inj["week"] == week) & inj["report_status"].isin(["Out", "Doubtful"])]
    out |= set(zip(ij["team"], norm_name(ij["full_name"])))
    # roster status for this week; a team with no rows for this week yet uses its latest earlier week
    ro = rosters[rosters["week"] <= week]
    ro = ro[ro["week"] == ro.groupby("team")["week"].transform("max")]
    ro = ro[ro["status"] == "RES"]
    out |= set(zip(ro["team"], norm_name(ro["full_name"])))

    past = snaps[snaps["week"] < week].copy()
    past["nm"] = norm_name(past["player"])
    team_games = past.groupby("team")["week"].nunique()
    detail = []
    for side, col in [("off", "offense_pct"), ("def", "defense_pct")]:
        pp = past[past[col] > 0]
        if side == "off":
            pp = pp[pp["position"] != "QB"]
        a = pp.groupby(["team", "nm"]).agg(player=("player", "first"), position=("position", "first"),
                                           games=("week", "nunique"), share=(col, "mean")).reset_index()
        a = a[a["share"] >= min_share].copy()
        a["frac"] = a["games"] / a["team"].map(team_games)
        a["out"] = [(t, n) in out for t, n in zip(a["team"], a["nm"])]
        a["lost"] = a["share"] * a["frac"] * a["out"]
        a["side"] = side
        detail.append(a)
    detail = pd.concat(detail, ignore_index=True)
    tot = detail.pivot_table(index="team", columns="side", values="lost", aggfunc="sum").reindex(columns=["off", "def"]).fillna(0.0)
    tot.columns = ["off_lost", "def_lost"]
    return tot, detail[detail["out"]].drop(columns=["nm", "out"])

LOST, lost_detail = missing_starters(snaps, inj, rosters, WEEK)
LOST = LOST.reindex(R.index).fillna(0.0)

wk = inj[inj["week"] == WEEK]
n_ruled = int(wk["report_status"].isin(["Out", "Doubtful"]).sum())
print(f"week {WEEK} injury report: {len(wk)} players listed | {n_ruled} Out or Doubtful | {int((wk['report_status'] == 'Questionable').sum())} Questionable | "
      f"{int(wk['report_status'].isna().sum())} with no game status (cleared, or not posted yet)")
if len(wk) == 0 or wk["report_status"].isna().mean() > 0.8:
    print(">> Game statuses are not posted yet. Re-run after Friday afternoon for a complete count.")
print("\nregular starters counted as unavailable (lost = playing time the ratings include that is now missing):")
print(lost_detail.sort_values(["team", "side", "lost"], ascending=[True, True, False])[
    ["team", "side", "player", "position", "games", "share", "lost"]].round(2).to_string(index=False))
print("\nteam totals, in starter-equivalents:")
print(LOST[(LOST["off_lost"] > 0) | (LOST["def_lost"] > 0)].round(2).sort_values("off_lost", ascending=False).to_string())

week 9 injury report: 362 players listed | 81 Out or Doubtful | 73 Questionable | 208 with no game status (cleared, or not posted yet)

regular starters counted as unavailable (lost = playing time the ratings include that is now missing):
team side              player position  games  share  lost
 ARI  def        Will Johnson       CB      2   0.90  0.22
 ARI  off     Kendrick Bourne       WR      8   0.55  0.55
 ARI  off       Paris Johnson       OL      4   1.00  0.50
 ARI  off          Jon Gaines       OL      4   0.90  0.45
 BAL  def     Jaylinn Hawkins        S      6   0.81  0.61
 BAL  def     Tavius Robinson       LB      6   0.72  0.54
 BAL  off       Jovaughn Gwyn       OL      3   0.91  0.34
 CAR  def       Bobby Okereke       LB      7   0.89  0.89
 CAR  def            Will Lee       CB      7   0.73  0.73
 CAR  def       Akayleb Evans       CB      5   0.73  0.52
 CAR  def     Jaelan Phillips       LB      5   0.64  0.46
 CAR  def         Jaycee Horn       CB      3   0.87 

## 8. Additions 3, 4 and 7: market starting point, wind and rain

**Market starting point.** For each team, the average points the betting line implied it would score, and allow, in its earlier games this season. It is the market's opinion of the team before this week. This week's line is not used.

**Wind.** Enter the forecast wind in mph for any outdoor game where it matters, keyed by the home team, for example `{"GB": 18, "BUF": 22}`. Anything at or under 10 mph does nothing, so only the windy games need an entry. Dome and closed-roof games are ignored.

**Rain.** List the home team of any outdoor game where rain is forecast at kickoff, for example `["CLE", "NE"]`. A reasonable rule: the hourly forecast at kickoff shows rain or showers, or a 50%+ chance of rain. Dome and closed-roof games are ignored. Check the forecast on Saturday night, when you run this.

In [8]:
def market_start(sched, week):
    """Average line-implied points for and against each team in games before `week`."""
    e = sched[(sched["week"] < week) & sched["spread_line"].notna() & sched["total_line"].notna()]
    h = pd.DataFrame({"team": e["home_team"], "mkt_for": e["total_line"] / 2 + e["spread_line"] / 2,
                      "mkt_against": e["total_line"] / 2 - e["spread_line"] / 2})
    a = pd.DataFrame({"team": e["away_team"], "mkt_for": e["total_line"] / 2 - e["spread_line"] / 2,
                      "mkt_against": e["total_line"] / 2 + e["spread_line"] / 2})
    return pd.concat([h, a]).groupby("team").mean()

def wind_over(roof, wind, floor=WIND_FLOOR):
    """mph above the floor in an outdoor game, else 0."""
    if str(roof) in ("dome", "closed") or pd.isna(wind):
        return 0.0
    return max(float(wind) - floor, 0.0)

MKT = market_start(sched, WEEK)

WIND_MPH = {
    # "GB": 18,
}
for team in WIND_MPH:
    if team not in set(sched.loc[sched["week"] == WEEK, "home_team"]):
        raise ValueError(f"WIND_MPH key {team!r} is not a home team in week {WEEK}")

RAIN_GAMES = [
    # "CLE",
]
for team in RAIN_GAMES:
    if team not in set(sched.loc[sched["week"] == WEEK, "home_team"]):
        raise ValueError(f"RAIN_GAMES entry {team!r} is not a home team in week {WEEK}")

def rain_flag(roof, game_id, home_team, rain):
    """1 if rain is expected in an outdoor game. rain: a list of home teams (live), or a dict game_id -> 0/1 (backtest)."""
    if str(roof) in ("dome", "closed"):
        return 0.0
    if isinstance(rain, dict):
        return float(rain.get(game_id, 0))
    return float(home_team in (rain or ()))

## 9. Additions 5 and 6: last season and pace

**Last season.** Early in the year a rating is built on few drives, so it is pulled toward league average. That treats last year's best and worst teams the same. This addition gives each offense and each defense a second pull, toward how it played last season: its full-season points per drive minus that season's league average. The pull fades as this season's games pile up. Its weight is 20 / (10.3 x weeks played + 20), which is 0.33 in week 5, 0.18 in week 10 and 0.11 in week 17.

**Pace.** Drives per game for the offense, plus drives per game faced by the opposing defense, each pulled toward the league average by adding 3 league-average games. More drives means more chances to score than a per-drive rating shows.

Both are automatic. Nothing needs to be typed in.

In [9]:
def build_drives_min(pbp):
    """Phase 2 rules, forecasting columns only: one row per qualifying offensive drive."""
    p = pbp[pbp["fixed_drive"].notna()].copy()
    p["drive_id"] = p["game_id"].astype(str) + "_" + p["fixed_drive"].astype(int).astype(str)
    qual = p["play_type"].fillna("").isin(["pass", "run"]) & (p["qb_kneel"].fillna(0) != 1) & (p["qb_spike"].fillna(0) != 1)
    p["is_qual"] = qual
    p["posteam_q"] = np.where(qual, p["posteam"], np.nan)
    p["defteam_q"] = np.where(qual, p["defteam"], np.nan)
    for c in ["epa", "success", "interception", "fumble_lost", "yardline_100"]:
        if c not in p.columns:
            p[c] = np.nan
    p["epa_q"] = np.where(qual, p["epa"].fillna(0.0), 0.0)
    p["success_q"] = np.where(qual, p["success"].fillna(0.0), 0.0)
    p["to_play"] = p["interception"].fillna(0) + p["fumble_lost"].fillna(0)
    p["yl_q"] = np.where(qual, p["yardline_100"], np.nan)
    g = p.groupby("drive_id", sort=False)
    out = g.agg(game_id=("game_id", "first"), season=("season", "first"), week=("week", "first"),
                posteam=("posteam_q", "first"), defteam=("defteam_q", "first"), fixed_drive=("fixed_drive", "first"),
                drive_result=("fixed_drive_result", "first"), qualifying_plays=("is_qual", "sum"),
                epa_sum=("epa_q", "sum"), success_plays=("success_q", "sum"), turnovers=("to_play", "sum"),
                deepest=("yl_q", "min")).reset_index()
    out = out[(out["qualifying_plays"] > 0) & out["posteam"].notna()].copy()
    res = out["drive_result"].fillna("")
    out["offensive_points"] = np.where(res == "Touchdown", 7, np.where(res == "Field goal", 3, 0))
    out["clock_expiration"] = res.str.startswith("End").astype(int)
    out["reached_20"] = (out["deepest"] <= 20).astype(int)
    out = out.join(drive_qb(pbp), on="drive_id").sort_values(["game_id", "posteam", "fixed_drive"])
    for c in ["qb_id", "qb"]:
        out[c] = out.groupby(["game_id", "posteam"])[c].ffill()
        out[c] = out.groupby(["game_id", "posteam"])[c].bfill()
    return out[out["clock_expiration"] == 0]

def season_ratings(dr):
    """dr: a full season of non-clock drives. Points per drive for each offense and each defense, as a gap from the league average."""
    lg = dr["offensive_points"].mean()
    return dr.groupby("posteam")["offensive_points"].mean() - lg, dr.groupby("defteam")["offensive_points"].mean() - lg

def last_season_weight(week):
    return FADE_K / (FADE_PER_WEEK * (week - 1) + FADE_K)

def pace_table(f, k=K_PACE):
    """f: non-clock drives so far this season. Returns (drives per game by team, league drives per game)."""
    lg = len(f) / f.groupby("posteam")["game_id"].nunique().sum()
    o = f.groupby("posteam").agg(n=("drive_id", "size"), g=("game_id", "nunique"))
    dd = f.groupby("defteam").agg(n=("drive_id", "size"), g=("game_id", "nunique"))
    return pd.DataFrame({"o_dpg": (o["n"] + k * lg) / (o["g"] + k), "d_dpg": (dd["n"] + k * lg) / (dd["g"] + k)}), lg

PREV_O, PREV_D = season_ratings(build_drives_min(prev_pbp))
PACE, LEAGUE_DPG = pace_table(f)
no_prev = sorted(set(R.index) - set(PREV_O.index))
if no_prev:
    print("no last-season rating found for:", no_prev, "(treated as league average)")

LSW = last_season_weight(WEEK)
ls = pd.DataFrame({"last_season_off": PREV_O.reindex(R.index), "last_season_def": PREV_D.reindex(R.index),
                   "drives_per_game": PACE["o_dpg"], "drives_faced_per_game": PACE["d_dpg"]})
ls["own_points_effect"] = COEF_ADV2["prev_o_w"] * LSW * ls["last_season_off"].fillna(0)
ls["opp_points_effect"] = COEF_ADV2["prev_d_w"] * LSW * ls["last_season_def"].fillna(0)
print(f"week {WEEK}: last season's weight is {LSW:.2f} | league drives per game this season: {LEAGUE_DPG:.2f}")
print("last_season_off / last_season_def = points per drive above (+) or below (-) the league last season. For a defense, + means it allowed more.")
print("own_points_effect = what last season adds to this team's own projection in Advanced 2.")
print("opp_points_effect = what this team's defense last season adds to its OPPONENT's projection.")
print(ls.sort_values("last_season_off", ascending=False).round(2).to_string())

week 9: last season's weight is 0.20 | league drives per game this season: 10.14
last_season_off / last_season_def = points per drive above (+) or below (-) the league last season. For a defense, + means it allowed more.
own_points_effect = what last season adds to this team's own projection in Advanced 2.
opp_points_effect = what this team's defense last season adds to its OPPONENT's projection.
     last_season_off  last_season_def  drives_per_game  drives_faced_per_game  own_points_effect  opp_points_effect
LA              0.80            -0.39            10.04                  10.31               0.60              -0.81
BUF             0.70            -0.03             9.94                  10.14               0.52              -0.06
IND             0.59             0.13            10.31                  10.13               0.44               0.27
NE              0.57            -0.37             9.76                   9.95               0.43              -0.77
SF              0.48

## 9b. Candidate additions (under test, not used live yet)

Five extra team ratings, all measured as a gap from league average (0 = average) and pulled toward average the same way the main ratings are. Offense ratings use the same quarterback weighting as the main offense rating.

- **EPA per play** and **success rate**, for each offense and each defense, `K_PLAYS` league-average plays added.
- **Recent form.** Points per drive with recent games counting more: a drive from n weeks before the last game counts `DECAY`^n.
- **Red-zone luck.** Points per drive on drives that had a snap inside the 20, compared with the league. Expected to pull back toward average, so the backtest should give it a negative weight.
- **Turnover luck.** Turnovers per drive (interceptions plus lost fumbles), compared with the league.

They are built from the play-by-play with the same function the backtest uses, so the live numbers and the replayed ones are defined identically. They only change a projection once `COEF_ADV3` is filled in from Section 12.


In [10]:
FORM_KEYS = ["epa", "sr", "recent", "rz", "to"]

def form_ratings(fr, starters, week, w_other=W_OTHER_QB):
    """fr: non-clock drives from weeks before `week` (build_drives_min output). Returns (offense table, defense table).
    Every column is a gap from league average, pulled toward 0."""
    x = fr.copy()
    mine = x["qb"] == x["posteam"].map(starters)
    x["wt"] = np.where(mine, 1.0, w_other)
    x["dw"] = DECAY ** (x["week"].max() - x["week"])
    plays = x["qualifying_plays"].sum()
    rz = x["reached_20"] == 1
    lg = {"epp": x["epa_sum"].sum() / plays, "sr": x["success_plays"].sum() / plays,
          "ppd": x["offensive_points"].mean(), "rz": x.loc[rz, "offensive_points"].mean(), "to": x["turnovers"].mean()}
    x["e_res"] = x["epa_sum"] - lg["epp"] * x["qualifying_plays"]
    x["s_res"] = x["success_plays"] - lg["sr"] * x["qualifying_plays"]
    x["p_res"] = x["offensive_points"] - lg["ppd"]
    x["rz_n"] = rz.astype(float)
    x["rz_res"] = np.where(rz, x["offensive_points"] - lg["rz"], 0.0)
    x["to_res"] = x["turnovers"] - lg["to"]

    def side(key, weighted):
        w = x["wt"] if weighted else pd.Series(1.0, index=x.index)
        s = pd.DataFrame({"n": w, "pl": w * x["qualifying_plays"], "e": w * x["e_res"], "s": w * x["s_res"],
                          "dn": w * x["dw"], "dp": w * x["dw"] * x["p_res"], "rn": w * x["rz_n"], "rr": w * x["rz_res"],
                          "to": w * x["to_res"], key: x[key]}).groupby(key).sum()
        return pd.DataFrame({"epa": s["e"] / (s["pl"] + K_PLAYS), "sr": s["s"] / (s["pl"] + K_PLAYS),
                             "recent": s["dp"] / (s["dn"] + K_OFF), "rz": s["rr"] / (s["rn"] + K_RZ),
                             "to": s["to"] / (s["n"] + K_TO)})
    return side("posteam", True), side("defteam", False)

FR_LIVE = build_drives_min(pbp)
FR_LIVE = FR_LIVE[FR_LIVE["week"] < WEEK]
FORM = form_ratings(FR_LIVE, starters, WEEK)
print("candidate ratings, gap from league average. Offense: + is better. Defense: + means it ALLOWED more (worse).")
print("epa = EPA per play | sr = success rate | recent = recent points per drive | rz = red-zone points per trip | to = turnovers per drive")
print(FORM[0].add_prefix("off_").join(FORM[1].add_prefix("def_")).sort_values("off_epa", ascending=False).round(3).to_string())


candidate ratings, gap from league average. Offense: + is better. Defense: + means it ALLOWED more (worse).
epa = EPA per play | sr = success rate | recent = recent points per drive | rz = red-zone points per trip | to = turnovers per drive
         off_epa  off_sr  off_recent  off_rz  off_to  def_epa  def_sr  def_recent  def_rz  def_to
posteam                                                                                          
SF         0.119   0.065       0.358   0.826   0.009   -0.027   0.001       0.013  -0.311  -0.002
BUF        0.118   0.036       0.725   0.576   0.022    0.045   0.019       0.068   0.289   0.002
JAX        0.113   0.030       0.473   0.135  -0.029   -0.076  -0.012      -0.410  -0.851   0.036
KC         0.085   0.028      -0.010  -0.243  -0.033   -0.020  -0.020      -0.146  -0.505   0.016
CHI        0.077   0.027       0.445   0.269   0.024   -0.015  -0.022      -0.198  -0.179  -0.017
DET        0.077   0.040       0.336   0.258  -0.043    0.077   0.017    

## 10. Project the week four ways, then save

**Saving is per game.** `SAVE_NOW` picks which games to lock in this run: a list of home teams, or `"all"`. A game that is already in the file is never changed. So the routine is:
- **Thursday**, before kickoff: `SAVE_NOW = ["BAL"]` (the Thursday home team). Only that game is saved.
- **Sunday morning**, after inactives: `SAVE_NOW = "all"`. Every game not saved yet is added, Monday night included.

Run the base notebook (WEEKLY_2) the same way, so both files have the same games. On a Thursday run the injury warning in Section 7 may fire because Sunday teams have not posted statuses yet. That is expected; the Thursday teams' statuses are final by Wednesday.

The `base` columns use the original formula and should match the base notebook when the starters are the same.

In [11]:
def week_inputs(week, R, sched, QB, LOST, MKT, wind_mph, prev_o, prev_d, pace, rain=None, form=None):
    """One row per team playing this week, with every input the four formulas use."""
    rows = []; lsw = last_season_weight(week)
    for g in sched[sched["week"] == week].itertuples():
        neutral = str(g.location) != "Home"
        wind = wind_mph.get(g.home_team, g.wind) if isinstance(wind_mph, dict) else g.wind
        for team, opp, h in [(g.home_team, g.away_team, 1), (g.away_team, g.home_team, -1)]:
            row = {"game_id": g.game_id, "week": week, "team": team, "opp": opp, "is_home": h == 1,
                         "off": R.loc[team, "off_rating"], "deff": R.loc[opp, "def_rating"],
                         "home": 0 if neutral else h, "new_qb": 1 - R.loc[team, "qb_share"],
                         "qb_prior": QB.loc[team, "qb_prior"], "qb_nohist": QB.loc[team, "qb_nohist"],
                         "off_lost": LOST.loc[team, "off_lost"], "def_lost": LOST.loc[opp, "def_lost"],
                         "wind_over": wind_over(g.roof, wind), "rain": rain_flag(g.roof, g.game_id, g.home_team, rain),
                         "mkt_for": MKT.loc[team, "mkt_for"], "opp_mkt_against": MKT.loc[opp, "mkt_against"],
                         "prev_o_w": lsw * prev_o.get(team, 0.0), "prev_d_w": lsw * prev_d.get(opp, 0.0),
                         "pace": pace.loc[team, "o_dpg"] + pace.loc[opp, "d_dpg"],
                         "market": g.total_line / 2 + h * g.spread_line / 2,
                         "already_played": bool(pd.notna(g.home_score)), "qb": R.loc[team, "expected_starter"]}
            if form is not None:                       # candidate additions: own offense and opponent's defense
                fo, fd = form
                for k in FORM_KEYS:
                    row["f_" + k] = fo[k].get(team, 0.0)
                    row["f_opp_" + k] = fd[k].get(opp, 0.0)
            rows.append(row)
    return pd.DataFrame(rows)

F = week_inputs(WEEK, R, sched, QB, LOST, MKT, WIND_MPH, PREV_O, PREV_D, PACE, RAIN_GAMES, form=FORM)
for team in RAIN_GAMES:
    print(f"rain entry {team}:", "counts" if F.loc[F['team'] == team, 'rain'].iloc[0] else "no effect (dome or closed roof)")
for team, mph in WIND_MPH.items():
    used = F.loc[F["team"] == team, "wind_over"].iloc[0]
    roof = sched.loc[(sched["week"] == WEEK) & (sched["home_team"] == team), "roof"].iloc[0]
    print(f"wind entry {team} {mph} mph:", f"counts as {used:.0f} mph over the floor" if used > 0 else f"no effect (roof: {roof}, floor: {WIND_FLOOR} mph)")
F["base"] = points(F, COEF)
F["adv"] = points(F, COEF_ADV)
F["adv2"] = points(F, COEF_ADV2)
F["adv_mkt"] = points(F, COEF_ADV_MKT)
if COEF_ADV3:                                  # only once Section 12 has produced it
    F["adv3"] = points(F, COEF_ADV3)
if BLEND_W:
    F["blend"] = BLEND_W["const"] + BLEND_W["model"] * F["adv_mkt"] + BLEND_W["market"] * F["market"]
EXTRA = [c for c in ["adv3", "blend"] if c in F.columns]

# what each addition is worth in the Advanced version, in points
F["pts_qb_history"] = COEF_ADV["qb_prior"] * F["qb_prior"] + COEF_ADV["qb_nohist"] * F["qb_nohist"]
F["pts_injuries"] = COEF_ADV["off_lost"] * F["off_lost"] + COEF_ADV["def_lost"] * F["def_lost"] + 0.0
F["pts_wind"] = COEF_ADV["wind_over"] * F["wind_over"] + 0.0
# and what the two newest additions are worth inside Advanced 2 (pace is measured against a league-average game)
F["pts_last_season"] = COEF_ADV2["prev_o_w"] * F["prev_o_w"] + COEF_ADV2["prev_d_w"] * F["prev_d_w"] + 0.0
F["pts_pace"] = COEF_ADV2["pace"] * (F["pace"] - 2 * LEAGUE_DPG) + 0.0
F["pts_rain"] = COEF_ADV_MKT["rain"] * F["rain"] + 0.0

print(f"WEEK {WEEK}: projected points per team, four versions and the betting line")
show = F[["team", "opp", "qb", "base", "adv", "adv2", "adv_mkt"] + EXTRA + ["market", "pts_qb_history", "pts_injuries", "pts_wind", "pts_last_season", "pts_pace", "pts_rain"]].copy()
show["adv2_minus_adv"] = show["adv2"] - show["adv"]
print(show.round(1).to_string(index=False))
print("\npts_qb_history / pts_injuries / pts_wind show what each addition contributes inside the Advanced number.")
print("pts_last_season / pts_pace show what last season and pace contribute inside Advanced 2.")
print("pts_rain shows what rain takes off in Advanced + market (Advanced 2 is about the same).")
print("They do not add up to the gap between versions exactly, because each formula also re-weights offense and defense.")

# the base columns should reproduce the original notebook's saved file when starters and settings match
orig = os.path.join(FOLDER, f"hidden_value_projections_week{WEEK:02d}.csv")
if os.path.exists(orig):
    o = pd.read_csv(orig)
    saved = pd.concat([o[["game_id", "home", "home_pts"]].set_axis(["game_id", "team", "saved_base"], axis=1),
                       o[["game_id", "away", "away_pts"]].set_axis(["game_id", "team", "saved_base"], axis=1)])
    chk = F[["game_id", "team", "base"]].merge(saved, on=["game_id", "team"], how="inner")
    diff = (chk["base"] - chk["saved_base"]).abs()
    print(f"\ncheck against the original notebook's saved week {WEEK} file: largest difference in base points = {diff.max():.2f}")
    if diff.max() > 0.05:
        print("   teams that differ (different starter, a neutral-site fix, or newer drive data):", sorted(chk.loc[diff > 0.05, "team"]))

# one row per game
hm = F[F["is_home"]].set_index("game_id"); aw = F[~F["is_home"]].set_index("game_id")
G = pd.DataFrame({"away": aw["team"], "home": hm["team"]})
for m in ["base", "adv", "adv2", "adv_mkt"] + EXTRA + ["market"]:
    G[m + "_margin"] = hm[m] - aw[m]
    G[m + "_total"] = hm[m] + aw[m]
print("\nhome margin and total by version (positive margin = home team):")
print(G.round(1).to_string(index=False))

WEEK 9: projected points per team, four versions and the betting line
team opp         qb  base  adv  adv2  adv_mkt  market  pts_qb_history  pts_injuries  pts_wind  pts_last_season  pts_pace  pts_rain  adv2_minus_adv
 BAL JAX  J.Fagnano  19.0 18.9  18.2     19.5    16.0            -1.2           0.5       0.0             -0.6      -0.1       0.0            -0.6
 JAX BAL T.Lawrence  25.9 26.2  27.0     25.8    24.5             0.1           1.1       0.0              0.4      -0.1       0.0             0.7
 ATL CIN    M.Penix  23.3 23.6  24.3     22.6    24.8             0.1           0.5       0.0              0.8      -0.2       0.0             0.8
 CIN ATL   J.Burrow  27.0 26.6  26.8     27.4    27.8             0.6          -0.1       0.0             -0.0      -0.1       0.0             0.2
 CAR DEN    B.Young  24.9 25.7  25.6     26.0    20.8            -1.0           2.2       0.0             -1.4       0.7       0.0            -0.1
 DEN CAR      B.Nix  21.9 25.8  26.5     25.6   

In [12]:
SAVE_NOW = "all"     # "all", or a list of home teams, e.g. ["BAL"] for a Thursday-only save

out_path = os.path.join(FOLDER, f"hidden_value_projections_advanced_week{WEEK:02d}.csv")
pick = F if SAVE_NOW == "all" else F[F["game_id"].isin(F.loc[F["is_home"] & F["team"].isin(SAVE_NOW), "game_id"])]
if SAVE_NOW != "all":
    bad = set(SAVE_NOW) - set(F.loc[F["is_home"], "team"])
    if bad:
        raise ValueError(f"SAVE_NOW has teams that are not home teams in week {WEEK}: {sorted(bad)}")
old = pd.read_csv(out_path) if os.path.exists(out_path) else None
have = set(old["game_id"]) if old is not None else set()
add = pick[~pick["game_id"].isin(have)].assign(saved_at=pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"))
kept = sorted(set(pick["game_id"]) & have)
if kept:
    print("already saved, NOT changed:", kept)
if add.empty:
    print("nothing new to save")
else:
    (add if old is None else pd.concat([old, add], ignore_index=True)).to_csv(out_path, index=False)
    late = sorted(add.loc[add["already_played"], "game_id"].unique())
    print(f"saved {add['game_id'].nunique()} game(s) to", out_path)
    if late:
        print("   these already had a final score, so they are saved but will not be graded:", late)
left = sorted(set(F["game_id"]) - have - set(add["game_id"]))
if left:
    print("not saved yet this week:", left)


already saved, NOT changed: ['2026_09_ARI_SEA', '2026_09_BUF_MIN', '2026_09_CIN_ATL', '2026_09_CLE_NO', '2026_09_DAL_IND', '2026_09_DEN_CAR', '2026_09_DET_MIA', '2026_09_GB_NE', '2026_09_HOU_LAC', '2026_09_JAX_BAL', '2026_09_LA_WAS', '2026_09_LV_SF', '2026_09_NYG_PHI', '2026_09_NYJ_KC', '2026_09_TB_CHI']
nothing new to save


## 11. Grade the four versions on the same games

This is the test of whether the additions are worth keeping. Games that already had a final score when they were saved are left out.

A week that was saved before Advanced 2 existed has no Advanced 2 number. So the first table grades every saved week without it, and the second grades all four versions on the weeks that have it.

Tied games are left out of the winners count. The last table is the running record of games where Advanced + market and the betting line pick different winners. In the 2016-2025 backtest the model's side won about 40% of those, so expect it to lose most of them.

In [13]:
def scorecard_adv(folder, sched):
    files = sorted(glob.glob(os.path.join(folder, "hidden_value_projections_advanced_week*.csv")))
    if not files:
        print("no saved advanced projections yet"); return None
    p = pd.concat([pd.read_csv(x) for x in files], ignore_index=True)
    p = p[~p["already_played"].astype(bool)]
    s = sched[sched["home_score"].notna()]
    act = pd.concat([s[["game_id", "home_team", "home_score"]].set_axis(["game_id", "team", "actual"], axis=1),
                     s[["game_id", "away_team", "away_score"]].set_axis(["game_id", "team", "actual"], axis=1)])
    t = p.merge(act, on=["game_id", "team"], how="inner")
    if t.empty:
        print("nothing to grade yet: no game that was projected in advance has a final score"); return None
    if "adv2" not in t.columns:
        t["adv2"] = np.nan

    def card(x, models, title):
        miss = x[models].sub(x["actual"], axis=0).abs()
        print(f"{title}: {len(x)} team scores (weeks {sorted(x['week'].unique().tolist())})")
        print("average miss per team, in points (lower is better):")
        print(miss.mean().round(2).to_string())
        print("average bias, projected minus actual (+ = projecting too many points):")
        print(x[models].sub(x["actual"], axis=0).mean().round(2).to_string())
        hm = x[x["is_home"]].set_index("game_id"); aw = x[~x["is_home"]].set_index("game_id")
        both = hm.index.intersection(aw.index)
        real = (hm.loc[both, "actual"] - aw.loc[both, "actual"])
        both = real[real != 0].index                        # ties have no winner to pick
        real = real.loc[both]
        print(f"winners picked ({len(both)} games with a winner):")
        for m in models:
            print(f"   {m:8s} {int((np.sign(hm.loc[both, m] - aw.loc[both, m]) == np.sign(real)).sum())} of {len(both)}")
        return miss

    def verdict(miss, n, a, b, label, expected):
        if n < 100:
            print("NOTE: under ~100 team scores this comparison is mostly noise. Do not decide yet.")
        else:
            d = (miss[a] - miss[b]).mean()
            print(f"{label}: {'ahead' if d > 0 else 'behind'} by {abs(d):.2f} points per team. The backtest expected about {expected} ahead.")

    t2 = t[t["adv2"].notna()]
    if len(t2) == len(t):                      # every graded week has all four versions: one table
        miss = card(t, ["base", "adv", "adv2", "adv_mkt", "market"], "EVERY GRADED WEEK")
        verdict(miss, len(t), "base", "adv", "Advanced against base", "0.08")
        if len(t) >= 100:
            verdict(miss, len(t), "adv", "adv2", "Advanced 2 against Advanced", "0.02 to 0.06")
    else:
        miss = card(t, ["base", "adv", "adv_mkt", "market"], "EVERY GRADED WEEK")
        verdict(miss, len(t), "base", "adv", "Advanced against base", "0.08")
        if t2.empty:
            print("\nno graded game has an Advanced 2 projection yet")
        else:
            print()
            miss2 = card(t2, ["base", "adv", "adv2", "adv_mkt", "market"], "WEEKS THAT HAVE ADVANCED 2")
            verdict(miss2, len(t2), "adv", "adv2", "Advanced 2 against Advanced", "0.02 to 0.06")

    for extra, label in [("adv3", "ADVANCED 3"), ("blend", "BLEND WITH THE LINE")]:
        if extra in t.columns and t[extra].notna().any():
            print()
            card(t[t[extra].notna()], ["base", "adv_mkt", extra, "market"], f"WEEKS THAT HAVE {label}")

    # games where Advanced + market and the line pick different winners
    hm = t[t["is_home"]].set_index("game_id"); aw = t[~t["is_home"]].set_index("game_id")
    both = hm.index.intersection(aw.index)
    g = pd.DataFrame({"away": aw.loc[both, "team"], "home": hm.loc[both, "team"],
                      "model_margin": hm.loc[both, "adv_mkt"] - aw.loc[both, "adv_mkt"],
                      "line_margin": hm.loc[both, "market"] - aw.loc[both, "market"],
                      "real_margin": hm.loc[both, "actual"] - aw.loc[both, "actual"]})
    dis = g[(np.sign(g["model_margin"]) != np.sign(g["line_margin"])) & (g["line_margin"] != 0) & (g["real_margin"] != 0)].copy()
    dis["model_pick"] = np.where(dis["model_margin"] > 0, dis["home"], dis["away"])
    dis["model_right"] = np.sign(dis["model_margin"]) == np.sign(dis["real_margin"])
    print(f"\nAdvanced + market vs the line, games they disagree on: model's pick won {int(dis['model_right'].sum())} of {len(dis)}"
          + " (backtest: about 40%)")
    if len(dis):
        print(dis[["away", "home", "model_pick", "model_margin", "line_margin", "real_margin", "model_right"]].round(1).to_string())
    return t

graded = scorecard_adv(FOLDER, sched)

EVERY GRADED WEEK: 84 team scores (weeks [5, 6, 7, 8])
average miss per team, in points (lower is better):
base       7.78
adv        7.60
adv2       7.55
adv_mkt    7.56
market     7.29
average bias, projected minus actual (+ = projecting too many points):
base       0.34
adv        0.49
adv2       0.54
adv_mkt    0.17
market    -0.64
winners picked (41 games with a winner):
   base     22 of 41
   adv      26 of 41
   adv2     29 of 41
   adv_mkt  27 of 41
   market   29 of 41
NOTE: under ~100 team scores this comparison is mostly noise. Do not decide yet.

Advanced + market vs the line, games they disagree on: model's pick won 4 of 10 (backtest: about 40%)
                away home model_pick  model_margin  line_margin  real_margin  model_right
game_id                                                                                  
2026_05_CHI_GB   CHI   GB         GB           1.3         -1.5         24.0         True
2026_05_LV_NE     LV   NE         LV          -2.8          3.

## 12. Appendix: where the constants come from (slow, optional)

Replays 2021-2025 with the same functions used above. For every week 4-18 it rates each team on earlier weeks only, builds the six additions from what was known before kickoff, and fits each formula with one season held out at a time. It pulls six seasons of play-by-play (one extra for the first last-season rating) and five of snap counts, injury reports and rosters, so it takes several minutes. Set `RUN_BACKTEST = True` to run it.

**Candidate additions (added Nov 9).** After the usual table it tests each Section 9b rating on top of Advanced + market + rain and on top of Advanced 2 + rain, then the blend with this week's line. It prints `COEF_ADV3` and `BLEND_W` and says whether each passed. A candidate passes only if it gains at least 0.02 points per team and is better in at least 4 of 5 seasons. The backtest now loads five more play-by-play columns, so it takes a little longer.

One difference from live use: in the replay, wind and rain are the weather recorded at the game, not a forecast. Rain = the play-by-play weather text mentions rain, showers or drizzle, in an outdoor or open-roof game.

In [ ]:
RUN_BACKTEST = False
BT_SEASONS = [2021, 2022, 2023, 2024, 2025]

def game_starters(pbp):
    """QB on each team's first dropback of each game = the starter known before kickoff. Returns name and ID."""
    p = pbp[(pbp["qb_dropback"] == 1) & pbp["id"].notna() & pbp["posteam"].notna()]
    return p.sort_values(["game_id", "play_id"]).drop_duplicates(["game_id", "posteam"]).set_index(["game_id", "posteam"])[["name", "id"]]

def rain_from_weather(pbp, sc):
    """Backtest rain flag per game, from the weather text in the play-by-play."""
    w = pbp.groupby("game_id")["weather"].first().fillna("").str.lower()
    wet = w.str.contains("rain|shower|drizzle") & ~w.str.contains("no rain|0% chance")
    return wet.astype(int).to_dict()

def backtest_season(s, fr, st, sc, hist, sn, ij, ro, w_other, prev_o, prev_d, weeks=range(4, 19), rain=None):
    """One row per team-game for one season, every input built from earlier weeks only."""
    rows = []
    ss = sc[sc["home_score"].notna()]
    for w in weeks:
        past, games = fr[fr["week"] < w], ss[ss["week"] == w]
        if games.empty or past.empty:
            continue
        names, ids = {}, {}
        for gm in games.itertuples():
            for team in (gm.home_team, gm.away_team):
                if (gm.game_id, team) in st.index:
                    names[team], ids[team] = st.loc[(gm.game_id, team), "name"], st.loc[(gm.game_id, team), "id"]
        r, _ = ratings(past, pd.Series(names), w_other, K_OFF, K_DEF)
        qb = qb_inputs(pd.Series(ids).reindex(r.index), hist)
        lost, _ = missing_starters(sn, ij, ro, w)
        lost = lost.reindex(r.index).fillna(0.0)
        mkt = market_start(sc, w)
        form = form_ratings(past, pd.Series(names), w, w_other)
        x = week_inputs(w, r, games, qb, lost, mkt, None, prev_o, prev_d, pace_table(past)[0], rain or {}, form=form)
        act = pd.concat([games.set_index(["game_id", "home_team"])["home_score"], games.set_index(["game_id", "away_team"])["away_score"]])
        x["pts"] = [act[(g, t)] for g, t in zip(x["game_id"], x["team"])]
        e = ss[ss["week"] < w]
        pf = pd.concat([e.set_index("home_team")["home_score"], e.set_index("away_team")["away_score"]]).groupby(level=0).mean()
        pa = pd.concat([e.set_index("home_team")["away_score"], e.set_index("away_team")["home_score"]]).groupby(level=0).mean()
        x["pf"], x["pa"] = x["team"].map(pf), x["opp"].map(pa)
        x["season"] = s
        rows.append(x)
    return pd.concat(rows, ignore_index=True)

def fit(X, cols):
    A = np.column_stack([np.ones(len(X))] + [X[c].values for c in cols])
    coef, *_ = np.linalg.lstsq(A, X["pts"].values, rcond=None)
    return coef

def loso(X, cols):
    """Leave-one-season-out predictions."""
    pred = pd.Series(np.nan, index=X.index)
    for s in X["season"].unique():
        tr, te = X[X["season"] != s], X[X["season"] == s]
        c = fit(tr, cols)
        pred[te.index] = np.column_stack([np.ones(len(te))] + [te[k].values for k in cols]) @ c
    return pred

if RUN_BACKTEST:
    all_stats = load_reg(nfl.load_player_stats, list(range(BT_SEASONS[0] - 3, BT_SEASONS[-1] + 1)), STAT_COLS)
    parts = []
    last = season_ratings(build_drives_min(load_reg(nfl.load_pbp, [BT_SEASONS[0] - 1], PBP_COLS)))   # the season before the first one replayed
    for y in BT_SEASONS:
        hp = load_reg(nfl.load_pbp, [y], PBP_COLS + ["weather"])
        fr, st = build_drives_min(hp), game_starters(hp)
        sc = load_reg(nfl.load_schedules, [y], SCHED_COLS)
        hist = qb_history(all_stats[all_stats["season"].between(y - 3, y - 1)])
        sn, ij, ro = load_reg(nfl.load_snap_counts, [y]), load_reg(nfl.load_injuries, [y]), load_reg(nfl.load_rosters_weekly, [y], ROSTER_COLS)
        parts.append(backtest_season(y, fr, st, sc, hist, sn, ij, ro, W_OTHER_QB, *last, rain=rain_from_weather(hp, sc)))
        last = season_ratings(fr)                                   # this season becomes "last season" for the next one
        print("replayed", y, len(parts[-1]), "team-games")
    X = pd.concat(parts, ignore_index=True).dropna(subset=["pts", "off", "deff", "pf", "pa"]).reset_index(drop=True)

    BASE = ["off", "deff", "home", "new_qb"]
    Q, I, M, WX = ["qb_prior", "qb_nohist"], ["off_lost", "def_lost"], ["mkt_for", "opp_mkt_against"], ["wind_over"]
    LS, PC, RN = ["prev_o_w", "prev_d_w"], ["pace"], ["rain"]
    versions = {"Base": BASE, "Base + QB history only": BASE + Q, "Base + injuries only": BASE + I,
                "Base + market start only": BASE + M, "Base + wind only": BASE + WX,
                "Base + last season only": BASE + LS, "Base + pace only": BASE + PC,
                "Advanced (1 + 2 + 4)": BASE + Q + I + WX, "Advanced + last season": BASE + Q + I + WX + LS,
                "Advanced 2 (+ last season + pace)": BASE + Q + I + WX + LS + PC,
                "Advanced + market (1 to 4)": BASE + Q + I + WX + M,
                "Advanced 2 + rain": BASE + Q + I + WX + LS + PC + RN, "Advanced + market + rain": BASE + Q + I + WX + M + RN}
    base_miss = (loso(X, BASE) - X["pts"]).abs()
    print(f"\n{len(X)} team-games, weeks 4-18, {BT_SEASONS[0]}-{BT_SEASONS[-1]}")
    print(f"{'version':34s} {'miss':>6s} {'vs base':>8s} {'seasons better':>15s} {'winner':>7s} {'vs spread':>10s}")
    for name, cols in versions.items():
        p = loso(X, cols); miss = (p - X["pts"]).abs()
        better = sum(base_miss[X["season"] == s].mean() > miss[X["season"] == s].mean() for s in X["season"].unique())
        gm = X.assign(p=p).sort_values("is_home").groupby("game_id").agg(pa=("p", "first"), ph=("p", "last"), ma=("market", "first"),
                                                                        mh=("market", "last"), a=("pts", "first"), h=("pts", "last"), n=("p", "size"))
        gm = gm[gm["n"] == 2]
        m, k, real = gm["ph"] - gm["pa"], gm["mh"] - gm["ma"], gm["h"] - gm["a"]
        win = (np.sign(m) == np.sign(real))[real != 0].mean()
        cover = np.sign(real - k); ok = (cover != 0) & (m != k)
        ats = (np.sign(m - k)[ok] == cover[ok]).mean()
        print(f"{name:34s} {miss.mean():6.2f} {base_miss.mean() - miss.mean():+8.2f} {str(better) + ' of 5' if name != 'Base' else '':>15s} {win:7.1%} {ats:10.1%}")
    print(f"{'Betting market':34s} {(X['market'] - X['pts']).abs().mean():6.2f} {base_miss.mean() - (X['market'] - X['pts']).abs().mean():+8.2f}")

    print("\nconstants fitted on all seasons (paste into Section 2 if you re-derive):")
    for label, cols in [("COEF", BASE), ("COEF_ADV", BASE + Q + I + WX), ("COEF_ADV2", BASE + Q + I + WX + LS + PC + RN),
                        ("COEF_ADV_MKT", BASE + Q + I + WX + M + RN)]:
        print(label, "=", {k: round(float(v), 3 if label in ("COEF_ADV2", "COEF_ADV_MKT") else 2) for k, v in zip(["const"] + cols, fit(X, cols))})

    # ---------- candidate additions (2026-11-09) ----------
    def per_season(miss):
        return miss.groupby(X["season"]).mean()
    def test_on(ref_name, ref_cols):
        ref_miss = (loso(X, ref_cols) - X["pts"]).abs()
        print(f"\non top of {ref_name}: {ref_miss.mean():.3f}")
        print(f"{'addition':22s} {'miss':>7s} {'gain':>7s} {'seasons better':>15s}")
        res = {}
        for name, cols in list(CAND.items()) + [("all five", sum(CAND.values(), []))]:
            miss = (loso(X, ref_cols + cols) - X["pts"]).abs()
            better = int((per_season(ref_miss) > per_season(miss)).sum())
            res[name] = (ref_miss.mean() - miss.mean(), better)
            print(f"{name:22s} {miss.mean():7.3f} {ref_miss.mean() - miss.mean():+7.3f} {str(better) + ' of 5':>15s}")
        return res

    CAND = {"EPA per play": ["f_epa", "f_opp_epa"], "Success rate": ["f_sr", "f_opp_sr"],
            "Recent form": ["f_recent", "f_opp_recent"], "Red-zone luck": ["f_rz", "f_opp_rz"],
            "Turnover luck": ["f_to", "f_opp_to"]}
    REF_MKT = BASE + Q + I + WX + M + RN            # the headline version
    REF_FB = BASE + Q + I + WX + LS + PC + RN       # football-only (Advanced 2 + rain)
    print("\nCANDIDATE ADDITIONS. Keep one only if it gains in at least 4 of 5 seasons; a gain under ~0.02 is noise.")
    res = test_on("Advanced + market + rain", REF_MKT)
    test_on("Advanced 2 + rain (football only)", REF_FB)

    keep = [n for n, (gain, better) in res.items() if n != "all five" and better >= 4 and gain >= 0.02]
    if keep:
        cols3 = REF_MKT + sum((CAND[n] for n in keep), [])
        m3 = (loso(X, cols3) - X["pts"]).abs()
        print(f"\nAdvanced 3 = Advanced + market + rain + {', '.join(keep)}: miss {m3.mean():.3f}")
        print("COEF_ADV3 =", {k: round(float(v), 3) for k, v in zip(["const"] + cols3, fit(X, cols3))})
        print("Paste that into Section 2 to start tracking Advanced 3 live.")
    else:
        print("\nNo candidate passed (4 of 5 seasons and at least 0.02). Leave COEF_ADV3 = None.")

    # ---------- blend with this week's line ----------
    Xb = X.assign(model=loso(X, REF_MKT))           # honest model predictions, each season from the other four
    pb = loso(Xb, ["model", "market"])
    mk_miss, b_miss = (X["market"] - X["pts"]).abs(), (pb - X["pts"]).abs()
    better = int((per_season(mk_miss) > per_season(b_miss)).sum())
    wb = fit(Xb, ["model", "market"])
    print(f"\nBLEND of Advanced + market + rain with this week's line: miss {b_miss.mean():.3f} vs line alone {mk_miss.mean():.3f} "
          f"({mk_miss.mean() - b_miss.mean():+.3f}, better in {better} of 5 seasons)")
    print("BLEND_W =", {"const": round(float(wb[0]), 3), "model": round(float(wb[1]), 3), "market": round(float(wb[2]), 3)})
    if better >= 4 and mk_miss.mean() - b_miss.mean() >= 0.02:
        print("The blend beats the line: paste BLEND_W into Section 2. The model then adds information the line does not have.")
    else:
        print("The blend does not reliably beat the line: leave BLEND_W = None.")


replayed 2021 448 team-games
replayed 2022 446 team-games
replayed 2023 448 team-games
replayed 2024 448 team-games
replayed 2025 448 team-games

2238 team-games, weeks 4-18, 2021-2025
version                              miss  vs base  seasons better  winner  vs spread
Base                                 7.50    +0.00                   61.6%      49.1%
Base + QB history only               7.49    +0.02          4 of 5   62.3%      49.5%
Base + injuries only                 7.46    +0.04          4 of 5   62.6%      49.1%
Base + market start only             7.46    +0.04          4 of 5   63.6%      48.9%
Base + wind only                     7.48    +0.03          4 of 5   61.6%      48.9%
Base + last season only              7.47    +0.03          5 of 5   62.0%      48.4%
Base + pace only                     7.50    +0.01          2 of 5   62.5%      49.1%
Advanced (1 + 2 + 4)                 7.42    +0.08          5 of 5   63.4%      49.8%
Advanced + last season               7.41